# 1. Race condition
* Execute the above code without private or firstprivate many times
* Observe that the value of `x` after the parallel region is not always 123 + 5 (the number of threads executing the region), even different across runs
* For example, with two threads, the following execution order may cause such a behavior
  1. thread A reads 123
  1. thread B reads 123
  1. thread A writes 124
  1. thread B reads 124
* A similar case occurs whenever a thread's read-followed-by-write is intervened by another thread's update
* More generally, the following situation is called a "race condition" and if there is a race condition in your program, it almost always means your program is broken
  * Two or more threads concurrently access the same variable, and
  * at least one of them writes to it
Here, "concurrently access" means these accesses are not guaranteed to be separated in time by a synchronization primitive

* In all but trivial parallel programs, threads need to communicate with each other to accomplish a task
* Threads _communicate_ by having one thread write to a variable and having another read it
* If we simply do it without any mechanism to guarantee that they are separated in time, it is a race

* Below, we describe three ways to _safely_ communicate among threads without making race conditions

  * `#pragma omp critical`
  * `#pragma omp atomic`
  * reduction


# 2. `#pragma omp critical`
* [#pragma omp critical](https://www.openmp.org/spec-html/5.2/openmpsu89.html#x120-4470002.17.1) guarantees the statement following the pragma is executed not overlapping in time

# <font color="green"> Problem 1 :  Apply `#pragma omp critical`</font>
* Execute the following program a few times and observe that the result is undeterministic and often not what we want (i.e., 123 + the number of threads)
* Then, add `#pragma omp critical` to the statement `x++` and see the result


In [ ]:
%%writefile omp_critical.c
#include <stdio.h>
#include <unistd.h>
#include <omp.h>

int main() {
  int x = 123;
  printf("before : x = %d\n", x);
#pragma omp parallel
  {
    int id = omp_get_thread_num();
    x++;
  }
  printf("after : x = %d\n", x);
  return 0;
}

In [ ]:
%%bash
clang -fopenmp omp_critical.c -o omp_critical
# nvc -mp omp_critical.c -o omp_critical

In [ ]:
%%bash
OMP_NUM_THREADS=100 ./omp_critical

# 3. `#pragma omp atomic`
* [#pragma omp atomic](https://www.openmp.org/spec-html/5.2/openmpsu95.html#x126-4840002.17.7) is similar to `#pragma omp critical` but its effect is slightly different and its applicability limited (see below)

# <font color="green"> Problem 2 :  Apply `#pragma omp atomic`</font>
* Add `#pragma omp atomic` to the statement `x++` and see the result


In [ ]:
%%writefile omp_atomic.c
#include <stdio.h>
#include <unistd.h>
#include <omp.h>

int main() {
  int x = 123;
  printf("before : x = %d\n", x);
#pragma omp parallel
  {
    int id = omp_get_thread_num();
    x++;
  }
  printf("after : x = %d\n", x);
  return 0;
}

In [ ]:
%%bash
clang -fopenmp omp_atomic.c -o omp_atomic
# nvc -mp omp_atomic.c -o omp_atomic

In [ ]:
%%bash
OMP_NUM_THREADS=100 ./omp_atomic

* The statement that follows this pragma cannot be an arbitrary expression
* See [atomic Construct](https://www.openmp.org/spec-html/5.2/openmpsu95.html#x126-4840002.17.7) for the spec
* Typically, it is an update to a variable, such as
```  
x += expr;
```  
* What is guaranteed by `#pragma omp atomic` is different from what `#pragma omp critical` guarantees
```
#pragma omp atomic
x += expr;
``` 
guarantees that the read and write to _x_ are never intervened by another update labeled `#pragma omp atomic` whereas
```
#pragma omp critical
x += expr;
``` 
guarantees that the entire statement `x += expr` does not overlap with another statement labeled critical.
* When applicable, `#pragma omp atomic` is more efficient than `#pragma omp critical` because the evaluation of expr can overlap
